# Barrido del criterio de alerta (H3 y H4)

Sobre los CSV de `inferencia_video.ipynb` (sin reprocesar video) se barre:

- τ ∈ {0,25 · 0,40 · 0,50 · 0,60}: umbral de confianza del detector.
- K/N ∈ {1/1 · 2/3 · 3/5 · 5/10}: criterio temporal. 1/1 equivale a no tener criterio temporal (alerta con la primera detección).
- Con y sin clasificador de recortes (`p_real ≥ UMBRAL_CLF`), si los CSV tienen la columna `p_real`.

Para cada combinación, sobre todos los clips:

- **Falsas alarmas por hora**: alertas en el clip sin evento y alertas antes del evento en los clips con evento, sobre el tiempo total sin evento observado.
- **Latencia de alerta**: segundos entre el evento visible (anotado a mano en `videos/clips.csv`) y la primera alerta posterior.
- **Tasa de eventos detectados**: clips con evento que terminan en alerta.

**H3:** comparar K/N = 1/1 contra el resto. **H4:** comparar con vs. sin clasificador a igual τ y K/N.

La lógica es la de `alerta.py` (la misma que usa la demo).

In [ ]:
import itertools
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import chi2

import alerta

REPO_ROOT = Path.cwd()
CLIPS_CSV = REPO_ROOT / "videos" / "clips.csv"
RESULTS_DIR = REPO_ROOT / "results"
FIG_DIR = REPO_ROOT / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

## 1. Configuración

- `DET_TAG`: qué detector (carpeta en `results/video/`, la misma que usó `inferencia_video.ipynb`).
- `FA_OBJETIVO`: falsas alarmas por hora máximas aceptables para elegir la configuración operativa (la de menor latencia que cumple el objetivo y detecta todos los eventos).

In [ ]:
DET_TAG = "yolo11n_640_neg_s42"
VIDEO_RESULTS = RESULTS_DIR / "video" / DET_TAG

TAUS = [0.25, 0.40, 0.50, 0.60]
KN = [(1, 1), (2, 3), (3, 5), (5, 10)]
UMBRAL_CLF = 0.50
FA_OBJETIVO = 1.0  # falsas alarmas por hora

assert VIDEO_RESULTS.exists(), f"No está {VIDEO_RESULTS}: correr inferencia_video.ipynb con ese detector"

## 2. Clips y detecciones

In [ ]:
clips = pd.read_csv(CLIPS_CSV)
datos = {}
for c in clips.itertuples():
    rutas = {k: VIDEO_RESULTS / f"{c.clip}_{k}.{ext}" for k, ext in (("detecciones", "csv"), ("frames", "csv"), ("meta", "json"))}
    if not all(p.exists() for p in rutas.values()):
        print(f"[!] {c.clip}: sin CSV de inferencia, se excluye")
        continue
    if c.tipo != "sin_evento" and pd.isna(c.t_evento_s):
        print(f"[!] {c.clip}: falta t_evento_s en clips.csv, se excluye (sin eso no hay latencia)")
        continue
    datos[c.clip] = {
        "tipo": c.tipo, "t_evento": c.t_evento_s,
        "det": pd.read_csv(rutas["detecciones"]), "frames": pd.read_csv(rutas["frames"]),
        "meta": json.loads(rutas["meta"].read_text(encoding="utf-8")),
    }

assert datos, "No hay clips utilizables"
assert any(d["tipo"] == "sin_evento" for d in datos.values()), "Hace falta al menos un clip sin_evento para medir falsas alarmas"

CON_CLF = all("p_real" in d["det"].columns for d in datos.values())
if CON_CLF:
    assert min(TAUS) >= 0.25, "p_real solo existe para conf > 0,25 (CLF_CONF_MIN de inferencia_video.ipynb)"
print("Con clasificador:", CON_CLF)

resumen_clips = pd.DataFrame([{
    "clip": k, "tipo": d["tipo"], "t_evento_s": d["t_evento"],
    "duracion_s": round(d["meta"]["duracion_s"], 1), "frames": len(d["frames"]), "detecciones": len(d["det"]),
} for k, d in datos.items()])
horas_sin_evento = sum(d["t_evento"] if d["tipo"] != "sin_evento" else d["meta"]["duracion_s"] for d in datos.values()) / 3600
print(f"Tiempo sin evento observado: {horas_sin_evento * 60:.1f} min")
resumen_clips

## 3. Barrido

`fa_h_cota95`: cota superior (95 %, Poisson exacta) de las falsas alarmas por hora. Con pocos minutos de video sin evento, "0 falsas alarmas" no significa 0 por hora: la cota dice hasta cuánto es compatible con lo observado, y es lo honesto para reportar.

In [ ]:
def cota95(n_eventos: int, horas: float) -> float:
    """Cota superior al 95 % (bilateral) de una tasa de Poisson con n eventos observados en `horas`."""
    return chi2.ppf(0.975, 2 * (n_eventos + 1)) / 2 / horas

umbrales = [None, UMBRAL_CLF] if CON_CLF else [None]
filas, por_clip = [], []
for tau, (k, n), u in itertools.product(TAUS, KN, umbrales):
    cfg = alerta.ConfigAlerta(tau=tau, k=k, n=n, umbral_clf=u)
    metricas = []
    for clip, d in datos.items():
        _, eventos = alerta.simular(d["frames"], d["det"], cfg)
        m = alerta.metricas_clip(eventos, d["meta"]["duracion_s"], d["t_evento"])
        metricas.append(m)
        por_clip.append({"config": cfg.nombre, "clip": clip, "tipo": d["tipo"], **m})
    r = alerta.resumir(pd.DataFrame(metricas))
    filas.append({
        "config": cfg.nombre, "tau": tau, "k": k, "n": n, "KN": f"{k}/{n}",
        "clasificador": u is not None, **r,
        "fa_h_cota95": cota95(r["falsas_alarmas"], horas_sin_evento),
    })

barrido = pd.DataFrame(filas)
por_clip = pd.DataFrame(por_clip)
barrido.to_csv(RESULTS_DIR / f"alerta_barrido_{DET_TAG}.csv", index=False)
por_clip.to_csv(RESULTS_DIR / f"alerta_barrido_por_clip_{DET_TAG}.csv", index=False)
barrido.round(3)

## 4. Falsas alarmas por hora vs. latencia

Un punto por combinación; color = τ, marcador = K/N, relleno = con clasificador. Abajo a la izquierda es mejor. Solo se dibujan las combinaciones que detectan **todos** los eventos (las que se pierden alguno no son una opción, aunque tengan cero falsas alarmas); las demás se listan debajo.

El eje x es *symlog*: lineal cerca de 0 (donde caen las configuraciones buenas) y logarítmico después.

In [ ]:
def frontera_pareto(df: pd.DataFrame) -> pd.DataFrame:
    """Puntos no dominados (menos falsas alarmas/hora y menos latencia)."""
    d = df.sort_values(["falsas_alarmas_por_hora", "latencia_media_s"])
    return d[d["latencia_media_s"] < d["latencia_media_s"].cummin().shift(fill_value=np.inf)]

validas = barrido[barrido["tasa_eventos_detectados"] == 1.0]
colores = dict(zip(TAUS, plt.cm.viridis(np.linspace(0, 0.85, len(TAUS)))))
marcadores = dict(zip([f"{k}/{n}" for k, n in KN], ["o", "s", "^", "D"]))

fig, ax = plt.subplots(figsize=(9, 5.5))
for r in validas.itertuples():
    ax.scatter(r.falsas_alarmas_por_hora, r.latencia_media_s, s=70, marker=marcadores[r.KN],
               color=colores[r.tau] if r.clasificador else "none", edgecolors=colores[r.tau], linewidths=1.5)
for clf, ls in ((False, "--"), (True, "-")):
    f = frontera_pareto(validas[validas["clasificador"] == clf])
    if len(f) > 1:
        ax.plot(f["falsas_alarmas_por_hora"], f["latencia_media_s"], ls=ls, color="k", lw=0.8, alpha=0.6,
                label=f"frontera {'con' if clf else 'sin'} clasificador")
for tau, c in colores.items():
    ax.scatter([], [], color=c, label=f"τ = {tau}")
for kn, m in marcadores.items():
    ax.scatter([], [], marker=m, color="gray", label=f"K/N = {kn}")
if CON_CLF:
    ax.scatter([], [], marker="o", color="none", edgecolors="gray", label="sin clasificador (hueco)")
ax.axvline(FA_OBJETIVO, color="tab:red", ls=":", lw=1, label=f"objetivo {FA_OBJETIVO}/h")
ax.set_xscale("symlog", linthresh=1)
ax.set_xlim(left=-0.3)
ax.set_xlabel("Falsas alarmas por hora")
ax.set_ylabel("Latencia media de alerta (s)")
ax.set_title(f"Criterio de alerta — {DET_TAG}")
ax.legend(fontsize=7, loc="upper left", bbox_to_anchor=(1.01, 1))
ax.grid(alpha=0.3)
fig.tight_layout()
for ext in ("png", "pdf"):
    fig.savefig(FIG_DIR / f"alerta_pareto_{DET_TAG}.{ext}", dpi=150, bbox_inches="tight")

perdidas = barrido[barrido["tasa_eventos_detectados"] < 1.0]
if len(perdidas):
    print("Configuraciones que no detectan todos los eventos (fuera de la figura):")
    print(perdidas[["config", "tasa_eventos_detectados", "falsas_alarmas_por_hora"]].to_string(index=False))

## 5. Lectura para H3 y H4

- **H3 (criterio temporal):** a cada τ, falsas alarmas/hora y latencia de 1/1 (sin criterio temporal) contra el resto.
- **H4 (clasificador):** a igual τ y K/N, sin vs. con clasificador.
- **Configuración operativa:** la de menor latencia media entre las que detectan todos los eventos y tienen falsas alarmas/hora ≤ `FA_OBJETIVO`.

In [ ]:
cols = ["falsas_alarmas_por_hora", "fa_h_cota95", "latencia_media_s", "tasa_eventos_detectados"]
h3 = barrido[~barrido["clasificador"]].pivot_table(index="tau", columns="KN", values=cols[:3]).round(2)
h3

In [ ]:
if CON_CLF:
    h4 = barrido.pivot_table(index=["tau", "KN"], columns="clasificador", values=cols).round(2)
    h4.columns = [f"{m}_{'con' if c else 'sin'}_clf" for m, c in h4.columns]
    display(h4)
else:
    print("Los CSV no tienen p_real: correr la sección 4 de inferencia_video.ipynb con el clasificador entrenado.")

In [ ]:
candidatas = validas[validas["falsas_alarmas_por_hora"] <= FA_OBJETIVO].sort_values(["latencia_media_s", "falsas_alarmas_por_hora"])
if len(candidatas):
    elegida = candidatas.iloc[0]
    print(f"Configuración operativa: {elegida['config']} | latencia media {elegida['latencia_media_s']:.1f} s "
          f"(máx {elegida['latencia_max_s']:.1f} s) | {elegida['falsas_alarmas_por_hora']:.2f} FA/h "
          f"(cota 95 %: {elegida['fa_h_cota95']:.1f}/h)")
    display(por_clip[por_clip["config"] == elegida["config"]])
else:
    print(f"Ninguna configuración detecta todos los eventos con ≤ {FA_OBJETIVO} FA/h: relajar el objetivo o revisar los clips.")